# Notebook 00 — Setup, Data Templates & Simulated Demo Data
**Project:** Intelligent Manufacturing & Price Estimation System (L&L Machinery internship)

> ⚠️ **DATA PROVENANCE WARNING.** Everything generated here is **SIMULATED** from arbitrary formulas.
> It is **not** L&L Machinery data. Prices, costs, lead times and model names are placeholders used only
> to test the code. Simulated results must **never** be presented as company performance.

**What this notebook does**
1. Checks the project folders and loads the reusable modules in `src/`
2. Writes **empty REAL-data templates** (headers only, no invented values) into `01_Data/`
3. Generates **SIMULATED** datasets (with deliberate data-quality problems) into `01_Data/simulated/`
4. Validates everything against the schemas

A model is only as defensible as its data definition, so we fix the data contract first and build cleaning, EDA and models on top.

## Step 1 — Locate the project

In [1]:
# --- Locate the project (works in Colab with Google Drive, or locally) ---
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    candidates = [
        "/content/drive/MyDrive/LL_MACHINERY_AI_PROJECT",
        "/content/drive/MyDrive/LL_MACHINERY_AI_PROJECT",
    ]
    PROJECT_ROOT = next((p for p in candidates if os.path.isdir(p)), candidates[0])
    if not os.path.isdir(PROJECT_ROOT):
        raise FileNotFoundError(
            "Project folder not found. Check that LL_MACHINERY_AI_PROJECT is inside "
            "Google Drive > MyDrive > LL JADO, or update PROJECT_ROOT."
        )
except ImportError:
    PROJECT_ROOT = os.path.abspath("..")          # running locally from 02_Notebooks
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/LL_MACHINERY_AI_PROJECT


## Step 2 — Load the reusable modules
| Module | Responsibility |
|---|---|
| `src/config.py` | Paths, random seed, provenance labels, **CSV schemas** |
| `src/validation.py` | Schema checks; blocks mixing REAL and SIMULATED rows |
| `src/data_generation.py` | **Simulated** data only (clearly labelled) |

In [2]:
import pandas as pd
from src.config import *
from src.validation import *
from src.data_generation import generate_specs_and_prices, generate_production

ensure_dirs()
for name, d in DIRS.items():
    print("✔", d.relative_to(PROJECT_ROOT))

✔ 01_Data
✔ 01_Data/simulated
✔ 01_Data/processed
✔ 02_Notebooks
✔ 03_Models
✔ 04_Evaluation
✔ 05_Dashboard


## Step 3 — Empty REAL-data templates
Header-only CSVs in `01_Data/`. These are the slots for **real** L&L records (with company permission). Nothing here is invented.
If a file already contains real rows, this step **does not overwrite it**.

In [3]:
templates = {
    FILES["specs"]: SPEC_SCHEMA,
    FILES["prices"]: PRICE_SCHEMA,
    FILES["production"]: PRODUCTION_SCHEMA,
}
for fname, schema in templates.items():
    path = DIRS["raw"] / fname
    if path.exists() and path.stat().st_size > 0 and len(pd.read_csv(path)) > 0:
        print("kept existing real data:", fname)
    else:
        pd.DataFrame(columns=list(schema)).to_csv(path, index=False)
        print("template written (headers only):", fname)

# Data dictionary: share with the company so they know what to export
rows = [(f, col, k, "yes" if req else "no", desc)
        for f, sch in templates.items() for col, (k, desc, req) in sch.items()]
data_dictionary = pd.DataFrame(rows, columns=["file", "column", "type", "required", "description"])
data_dictionary.to_csv(DIRS["reports"] / "data_dictionary.csv", index=False)
data_dictionary.head(8)

template written (headers only): machine_specifications.csv
template written (headers only): historical_prices.csv
template written (headers only): production_records.csv


,file,column,type,required,description
0,machine_specifications.csv,quote_id,str,yes,Unique quotation ID (links to historical_prices)
1,machine_specifications.csv,machine_type,str,yes,CNC or Conventional
2,machine_specifications.csv,model_series,str,yes,Model / series name
3,machine_specifications.csv,swing_over_bed_mm,num,yes,Swing over bed (mm)
4,machine_specifications.csv,distance_between_centers_mm,num,yes,Distance between centers (mm)
5,machine_specifications.csv,spindle_bore_mm,num,yes,Spindle bore diameter (mm)
6,machine_specifications.csv,spindle_motor_kw,num,yes,Main spindle motor power (kW)
7,machine_specifications.csv,chuck_size_inch,num,no,Chuck size (inch)


## Step 4 — SIMULATED demonstration data
Why simulate? To build and test the whole pipeline **before** real data arrives.
Why dirty? Real data has missing values, duplicates and outliers; a few are injected so Notebook 01 (cleaning/EDA) has real work to do.

**Safeguards:** file names start with `SIMULATED_`, every row has `data_source = "SIMULATED"`, and `save_dataset()` refuses mismatches.

In [4]:
sim_specs, sim_prices = generate_specs_and_prices(n=400)
sim_prod = generate_production(n=500)

save_dataset(sim_specs,  DIRS["simulated"] / "SIMULATED_machine_specifications.csv")
save_dataset(sim_prices, DIRS["simulated"] / "SIMULATED_historical_prices.csv")
save_dataset(sim_prod,   DIRS["simulated"] / "SIMULATED_production_records.csv")

print("specs", sim_specs.shape, "| prices", sim_prices.shape, "| production", sim_prod.shape)
print({k: provenance_summary(v).to_dict() for k, v in
       {"specs": sim_specs, "prices": sim_prices, "production": sim_prod}.items()})

specs (405, 15) | prices (405, 9) | production (500, 15)
{'specs': {'SIMULATED': 405}, 'prices': {'SIMULATED': 405}, 'production': {'SIMULATED': 500}}


## Step 5 — Validate against the schemas
The duplicate-ID issues are **expected**: the simulator injected them on purpose.

In [5]:
report = pd.concat([
    validate_schema(sim_specs,  SPEC_SCHEMA,       "SIM specs"),
    validate_schema(sim_prices, PRICE_SCHEMA,      "SIM prices"),
    validate_schema(sim_prod,   PRODUCTION_SCHEMA, "SIM production"),
], ignore_index=True)
report if len(report) else "No schema issues"

,dataset,column,issue
0,SIM specs,quote_id,5 duplicate IDs
1,SIM prices,quote_id,5 duplicate IDs


## Step 6 — First look

In [6]:
display(sim_specs.head(3)); display(sim_prices.head(3))
print("Missing values (specs):  ", sim_specs.isna().sum()[lambda s: s > 0].to_dict())
print("Missing values (prices): ", sim_prices.isna().sum()[lambda s: s > 0].to_dict())
print("\nProduction status distribution:")
print(sim_prod["production_status"].value_counts())

,quote_id,machine_type,model_series,swing_over_bed_mm,distance_between_centers_mm,spindle_bore_mm,spindle_motor_kw,chuck_size_inch,turret_stations,live_tooling,bar_feeder,chip_conveyor,tailstock_type,controller,data_source
0,SIMQ-00001,Conventional,SIM-CV-2,420,3000,52,22.0,15.0,0,0,0,0,hydraulic,NONE,SIMULATED
1,SIMQ-00002,CNC,SIM-CNC-A,500,750,80,11.0,15.0,10,0,0,1,programmable,CTRL-A,SIMULATED
2,SIMQ-00003,Conventional,SIM-CV-2,600,1000,80,22.0,15.0,0,0,0,1,programmable,NONE,SIMULATED


,quote_id,quote_date,customer_region,material_cost,labor_hours,manufacturing_cost,quoted_price,final_selling_price,data_source
0,SIMQ-00001,2024-07-31,Region_1,866.0,295.0,1439.4,1955.0,1899.1,SIMULATED
1,SIMQ-00002,2024-11-11,Region_1,1254.7,263.5,1815.0,2381.3,2296.9,SIMULATED
2,SIMQ-00003,2023-01-26,Region_2,804.6,227.5,1260.4,1658.3,1571.0,SIMULATED


Missing values (specs):   {'chuck_size_inch': 12}
Missing values (prices):  {'labor_hours': 16}

Production status distribution:
production_status
Machining               164
Completed               130
Assembly                 95
Material preparation     51
Delayed                  49
Testing                  11
Name: count, dtype: int64


## Methodology notes (for your defence)
1. **Target.** Price model target = `final_selling_price`. `quoted_price` is *excluded* from features later: it is nearly the answer itself (**target leakage**).
2. **Cost as a feature.** `manufacturing_cost` is only a legitimate input if it is known *at estimation time*. We will test models with and without it.
3. **Circularity.** Simulated price comes from a formula, so a model only recovers that formula. Simulated results validate the **code**, not the **business**.
4. **Two ML tasks.** Price = regression. Production status = multi-class classification. Completion date = regression on lead time.
5. **Time-aware split** (Notebook 02): train on earlier quotations, test on later ones.

**Next:** `01_data_exploration.ipynb` (cleaning and EDA).